# 06 – Causal analysis with DoWhy

**Goal:** check whether key risk factors **cause** higher 10-year CHD risk, not just appear alongside it.

**Why:** SHAP shows what the model uses, but correlation can be misleading. Example: smokers in this data are younger, which hides the true effect of smoking. DoWhy adjusts for these **confounders** and then tests whether the estimate holds up.

**Factors tested:** smoking, high blood pressure (systolic ≥ 140), diabetes.

## Step 1 – Load data

Causal analysis needs complete records, so rows with missing values are dropped (4,238 → 3,749).
`high_BP` = systolic blood pressure ≥ 140 mmHg (standard hypertension cut-off).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import logging; logging.disable(logging.WARNING)
import numpy as np, pandas as pd
from dowhy import CausalModel

np.random.seed(42)

df = pd.read_csv("../data/framingham.csv")
df.loc[(df.currentSmoker == 0) & (df.cigsPerDay.isna()), "cigsPerDay"] = 0
df = df.drop(columns=["education"]).dropna().reset_index(drop=True)
df["high_BP"] = (df["sysBP"] >= 140).astype(int)
print(df.shape)

(3749, 16)


## Step 2 – Estimate causal effects

For each factor:
1. Define the **confounders**: other factors that affect both the risk factor and heart disease (e.g. age, sex)
2. DoWhy builds a causal model and adjusts for them (backdoor adjustment, linear regression)
3. Compare the **naive difference** (raw comparison) with the **causal effect** (after adjustment)

Effects are in **percentage points** of 10-year CHD risk.

In [2]:
def causal_effect(treatment, confounders):
    d = df.copy()
    d[treatment] = d[treatment].astype(bool)
    model = CausalModel(data=d, treatment=treatment, outcome="TenYearCHD", common_causes=confounders)
    estimand = model.identify_effect(proceed_when_unidentifiable=True)
    estimate = model.estimate_effect(estimand, method_name="backdoor.linear_regression",
                                     confidence_intervals=True)
    lo, hi = np.array(estimate.get_confidence_intervals()).flatten()
    naive = d.loc[d[treatment], "TenYearCHD"].mean() - d.loc[~d[treatment], "TenYearCHD"].mean()
    return model, estimand, estimate, {
        "factor": treatment,
        "naive_difference_%": naive * 100,
        "causal_effect_%": estimate.value * 100,
        "95%_CI_low": lo * 100, "95%_CI_high": hi * 100,
    }

results = []
m_smoke, e_smoke, est_smoke, r = causal_effect("currentSmoker",
    ["age", "male", "sysBP", "totChol", "BMI", "diabetes", "glucose", "BPMeds"]); results.append(r)
m_bp, e_bp, est_bp, r = causal_effect("high_BP",
    ["age", "male", "currentSmoker", "cigsPerDay", "totChol", "BMI", "diabetes", "glucose"]); results.append(r)
m_dm, e_dm, est_dm, r = causal_effect("diabetes",
    ["age", "male", "currentSmoker", "sysBP", "totChol", "BMI", "BPMeds"]); results.append(r)

pd.DataFrame(results).round(1)

,factor,naive_difference_%,causal_effect_%,95%_CI_low,95%_CI_high
0,currentSmoker,1.6,4.7,2.4,7.0
1,high_BP,13.6,7.9,5.3,10.5
2,diabetes,20.6,12.8,6.0,19.6


## Step 3 – Why does smoking look weak at first?

In [3]:
df.groupby("currentSmoker")["age"].mean().round(1)

currentSmoker
0    51.3
1    47.7
Name: age, dtype: float64

## Step 4 – Robustness checks (refutation tests)

Three tests to see if the estimates are trustworthy:
- **Placebo:** replace the real factor with random noise → effect should drop to **~0**
- **Random confounder:** add a fake random confounder → effect should **stay the same**
- **80% subset:** re-estimate on random 80% samples → effect should **stay similar**

In [4]:
checks = []
for name, m, e, est in [("currentSmoker", m_smoke, e_smoke, est_smoke),
                        ("high_BP", m_bp, e_bp, est_bp),
                        ("diabetes", m_dm, e_dm, est_dm)]:
    placebo   = m.refute_estimate(e, est, method_name="placebo_treatment_refuter",
                                  placebo_type="permute", num_simulations=50)
    random_cc = m.refute_estimate(e, est, method_name="random_common_cause", num_simulations=50)
    subset    = m.refute_estimate(e, est, method_name="data_subset_refuter",
                                  subset_fraction=0.8, num_simulations=50)
    checks.append({"factor": name,
                   "original_%": est.value * 100,
                   "placebo_% (should be ~0)": placebo.new_effect * 100,
                   "random_confounder_% (should match)": random_cc.new_effect * 100,
                   "80%_subset_% (should match)": subset.new_effect * 100})

pd.DataFrame(checks).round(2)

,factor,original_%,placebo_% (should be ~0),random_confounder_% (should match),80%_subset_% (should match)
0,currentSmoker,4.73,0.04,4.73,4.71
1,high_BP,7.90,0.42,7.90,7.79
2,diabetes,12.77,0.13,12.78,12.29


## Summary

| Factor | Naive difference | Causal effect (95% CI) | Refutation tests |
|---|---|---|---|
| Smoking | +1.6 pts | **+4.7 pts** (2.4 – 7.0) | Passed |
| High BP (≥140) | +13.6 pts | **+7.9 pts** (5.3 – 10.5) | Passed |
| Diabetes | +20.6 pts | **+12.8 pts** (6.0 – 19.6) | Passed |

- All three factors have a **positive causal effect** on 10-year CHD risk after adjusting for confounders
- **Smoking's effect was hidden** by age (smokers are younger) – adjustment triples it
- **High BP and diabetes were overstated** in raw comparisons, because those patients are also older
- All estimates passed placebo, random-confounder and subset refutation tests
- This supports the SHAP findings (notebook 01) with causal evidence, strengthening the explainability claim

**Limitation:** causal conclusions assume all major confounders are measured. Unmeasured factors (diet, exercise, family history) could still bias estimates.